In [2]:
import numpy as np
import matplotlib.pyplot as plt
from nnfs.datasets import spiral_data

np.set_printoptions(precision=2)

In [16]:
class DenseLayer:
    def __init__(self, dim_in, dim_out):
        # here we are creating a transposed metrix to avoid transpose at each evaluation of a layer
        # therfore here columns represent the neurons and rows represent the nth input wights of each
        # neuron
        self.weights = np.random.randn(dim_in, dim_out)
        # using zero as bias for sometime
        self.bias = np.zeros((1, dim_out))
    
    def forward(self, inputs):
        self.output = np.dot(inputs, self.weights) + self.bias
        return self.output

class ReLU():
    def forward(self, x):
        self.output = np.maximum(0, x)
        return self.output
    
class Softmax():
    def forward(self, x):
        ex = np.exp(x - x.max(axis=1, keepdims=1))
        self.output = ex/ex.sum(axis=1, keepdims=1)
        return self.output

Loss function are used to verify how much accurate our output predictions
For classification problems we can use a categorical cross entrophy loss

The idea is that we have a softmax activation at the end of last layer. The outout of each neuron is 
takesn as probability of a particular outcome.

categorical cross entrohy = -1 * truelabel value * log(probability)


In [43]:
class CategoricalCrossEntrophyLoss():
    def forward(self, y_val, y_true):
        
        if len(y_true.shape) == 1:
            likelihood = y_val[np.arange(len(y_val)), np.array(y_true)]
        elif len(y_true.shape) == 2: # using one hot encoding
            likelihood = np.sum(y_val * y_true, axis=1) # elementwise multiplicaiton

        log_likelihood = np.log(np.clip(likelihood, 1e-7, 1-1e-7))
        neg_log_likelihood = -1 * log_likelihood
        return neg_log_likelihood


class Loss():
    def calculate(self, lossess):
        return np.average(lossess)

In [31]:
X, Y = spiral_data(samples=10, classes=3)
layer = DenseLayer(2, 3)
softmax = Softmax()
probs = softmax.forward(layer.forward(X))


In [24]:
loss_values = -1 * np.log(np.clip(probs[np.arange(len(X)), np.array(Y)], 1e-7, 1-1e-7))
np.average(loss_values)

1.4086632675391721

In [45]:
loss = CategoricalCrossEntrophyLoss()
nll = loss.forward(probs, Y)
loss_f = Loss()
loss_f.calculate(nll)

1.1806945502257091

In [110]:
# explanation
probs[:5] # each value in the row corresponds to probability of each class of output

array([[0.33, 0.33, 0.33],
       [0.33, 0.37, 0.31],
       [0.33, 0.39, 0.28],
       [0.37, 0.33, 0.3 ],
       [0.31, 0.23, 0.46]])

In [111]:
Y[:5]

array([0, 0, 0, 0, 0], dtype=uint8)

In [112]:
probs[:5][[0, 1, 2, 3, 4], Y[:5]]

array([0.33, 0.33, 0.33, 0.37, 0.31])

In [113]:
probs[:5][np.arange(5), Y[:5]]

array([0.33, 0.33, 0.33, 0.37, 0.31])

In [114]:
# another way is to create one hot encoding for output classess and doing a scalar multiplication
one_hot_ys = np.zeros((len(Y), Y.max()+1))
one_hot_ys[np.arange(len(Y)), Y ] = 1

In [115]:
probs * one_hot_ys

array([[0.33, 0.  , 0.  ],
       [0.33, 0.  , 0.  ],
       [0.33, 0.  , 0.  ],
       [0.37, 0.  , 0.  ],
       [0.31, 0.  , 0.  ],
       [0.28, 0.  , 0.  ],
       [0.26, 0.  , 0.  ],
       [0.34, 0.  , 0.  ],
       [0.43, 0.  , 0.  ],
       [0.39, 0.  , 0.  ],
       [0.  , 0.33, 0.  ],
       [0.  , 0.3 , 0.  ],
       [0.  , 0.32, 0.  ],
       [0.  , 0.42, 0.  ],
       [0.  , 0.43, 0.  ],
       [0.  , 0.5 , 0.  ],
       [0.  , 0.24, 0.  ],
       [0.  , 0.2 , 0.  ],
       [0.  , 0.58, 0.  ],
       [0.  , 0.61, 0.  ],
       [0.  , 0.  , 0.33],
       [0.  , 0.  , 0.3 ],
       [0.  , 0.  , 0.35],
       [0.  , 0.  , 0.42],
       [0.  , 0.  , 0.45],
       [0.  , 0.  , 0.4 ],
       [0.  , 0.  , 0.18],
       [0.  , 0.  , 0.18],
       [0.  , 0.  , 0.22],
       [0.  , 0.  , 0.58]])

In [41]:
nll = -np.average(np.log(np.clip(np.sum(probs * one_hot_ys, axis=1), 1e-7, 1-1e-7)))

In [44]:
loss = CategoricalCrossEntrophyLoss()
nll = loss.forward(probs, one_hot_ys)
loss_f = Loss()
loss_f.calculate(nll)

1.1806945502257091

In [42]:
nll

1.1806945502257091

In [125]:
probs, np.max(probs[:5], axis=1, keepdims=1)

(array([[0.33, 0.33, 0.33],
        [0.33, 0.37, 0.31],
        [0.33, 0.39, 0.28],
        [0.37, 0.33, 0.3 ],
        [0.31, 0.23, 0.46],
        [0.28, 0.28, 0.44],
        [0.26, 0.49, 0.24],
        [0.34, 0.49, 0.17],
        [0.43, 0.32, 0.25],
        [0.39, 0.15, 0.45],
        [0.33, 0.33, 0.33],
        [0.33, 0.3 , 0.36],
        [0.31, 0.32, 0.37],
        [0.3 , 0.42, 0.28],
        [0.34, 0.43, 0.23],
        [0.29, 0.5 , 0.22],
        [0.4 , 0.24, 0.36],
        [0.27, 0.2 , 0.53],
        [0.24, 0.58, 0.18],
        [0.27, 0.61, 0.12],
        [0.33, 0.33, 0.33],
        [0.33, 0.36, 0.3 ],
        [0.36, 0.29, 0.35],
        [0.33, 0.25, 0.42],
        [0.31, 0.24, 0.45],
        [0.28, 0.32, 0.4 ],
        [0.32, 0.5 , 0.18],
        [0.36, 0.46, 0.18],
        [0.41, 0.37, 0.22],
        [0.31, 0.12, 0.58]]),
 array([[0.33],
        [0.37],
        [0.39],
        [0.37],
        [0.46]]))

In [28]:
len(Y.shape)

1